# 04 - Centrality Analysis

מחברת זו מדרגת את התחנות של רשת התחבורה הציבורית בישראל לפי מספר מדדי מרכזיות
(centrality), ובוחנת האם המדדים השונים מסכימים זה עם זה. אנו מחשבים degree ו-degree
משוקלל (נפח שירות), PageRank משוקלל על גרף הנסיעות המכוון (חשיבות זרימה), betweenness
מקורב על הרכיב הקשיר הגדול ביותר (חשיבות כצומת מעבר / גישור) ו-harmonic centrality
(נגישות). לאחר מכן אנו משווים בין הדירוגים באמצעות מטריצת מתאם Spearman, ובעיקר -
מודדים עד כמה רועש בפועל אומדן ה-betweenness המדגמי, לפני שמישהו בונה מעליו סף של
"תחנה קריטית".

**שאלת המחקר.** האם מדדים מקומיים וזולים (degree, degree משוקלל, PageRank) מזהים את
אותן תחנות שמזהה המדד הגלובלי והיקר (betweenness)? אם כן, המדדים הזולים מהווים proxy
טוב; אם לא, קיימות ברשת תחנות בעלות חשיבות מבנית שנושאות תנועה מועטה.

**קלט**
- `outputs/nb/02_graph_construction/` - טבלת הצמתים וטבלת הקשתות המכוונות שהופקו
  במחברת `02_graph_construction` (מאפייני תחנה; `from_stop`, `to_stop`, תדירות נסיעות
  לכל מקטע).
- אין צורך בקובץ GTFS גולמי. הקובץ `stop_times.txt` (816 MB) נדרש רק לבדיקת שפיות
  אופציונלית בסוף המחברת, ורק אם הוא כבר קיים על הדיסק.

**פלט** (הכול תחת `outputs/nb/04_centrality_analysis/`)
- `tables/stop_metrics.csv` - שורה אחת לכל תחנה עם כל מדדי המרכזיות.
- `tables/top_degree.csv`, `top_weighted_degree.csv`, `top_pagerank.csv`,
  `top_approx_betweenness.csv`, `top_approx_harmonic.csv` - דירוגי top-N.
- `tables/centrality_correlation_spearman.csv` - מטריצת Spearman בין המדדים.
- `tables/betweenness_stability.csv` - מידת השינוי בדירוג ה-betweenness כאשר משתנה רק
  המדגם האקראי של צמתי המקור.
- `figures/` - תרשימי עמודות top-N לכל מדד, מפת חום של המתאמים, תרשימי פיזור של degree
  מול שאר המדדים, מפה של תחנות ה-betweenness המובילות, ותרשים הפיזור של יציבות
  seed מול seed.

**הערת יושרה מקדימה.** ה-betweenness כאן הוא קירוב מבוסס מדגם של *k* מקורות
(ברירת מחדל `K_BETWEENNESS = 300` מקורות שנדגמו מתוך כ-30,000 צמתים, כלומר כ-1%
מהמקורות האפשריים). האומדן רועש, במיוחד בזנב ההתפלגות, וכל כלל במורד הזרם מסוג
"תחנה קריטית" - למשל סף p90 על betweenness - יורש את הרעש הזה. אנו מכמתים זאת במקום
להסתיר זאת.

## 1. אתחול סביבת העבודה

התא הבא מאפשר להריץ את המחברת הן מקומית והן ב-Google Colab. הוא מאתר את שורש
המאגר על ידי טיפוס כלפי מעלה מהתיקייה הנוכחית בחיפוש אחר תיקיית נתוני ה-GTFS, ואם אינו
מוצא אותה (כלומר אנו על מכונת Colab חדשה) הוא משכפל את המאגר. כמו כן הוא מתקין רק
חבילות חסרות - דבר אינו מותקן מחדש אם הוא כבר זמין. כל התאים שבהמשך מניחים
ש-`REPO`, `DATA` ו-`OUT` קיימים.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ייבוא ספריות, תיקיות פלט וקבועים ניתנים לכוונון

כל הפרמטרים השולטים בזמן הריצה מרוכזים בתא יחיד זה, כך שהבודק יכול להקטין אותם מבלי
לקרוא את שאר המחברת.

- `K_BETWEENNESS = 300` - מספר צמתי המקור שנדגמים אקראית עבור ה-betweenness המקורב.
  חישוב betweenness מדויק על גרף זה מחייב BFS מכל אחד מכ-30,000 הצמתים (שעות). עם
  `k = 300` ריצה אורכת בערך 1-3 דקות.
- `RUN_STABILITY_CHECK` - מריץ את ה-betweenness *פעם נוספת* עם seed אקראי שונה, כדי
  שנוכל למדוד עד כמה הדירוג תלוי במדגם. הדבר מכפיל בקירוב את עלות ה-betweenness; ניתן
  להגדיר `False` אם ממהרים, אך טיעון היושרה בסעיף 11 מסתמך על כך.
- `HARMONIC_SAMPLES = 300` - גם harmonic centrality נאמד ממקורות שנדגמו (כל מקור עולה
  BFS אחד). יש להגדיר `None` עבור החישוב המדויק, שהוא BFS אחד לכל צומת ואורך עשרות דקות.
- `TOP_N = 15` - אורך טבלאות הדירוג ותרשימי העמודות.

מחברת זו כותבת אך ורק לתיקיית השלב שלה, `outputs/nb/04_centrality_analysis`. היא לעולם
אינה נוגעת ב-`outputs/tables`, ב-`outputs/figures` או ב-`outputs/rail`, המכילות את
התוצאות המצוטטות בדוח הכתוב.

In [ ]:
_ensure("networkx", "pandas", "numpy", "matplotlib")

import csv
import json
import pickle
import random
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt

# ---------------- tunable constants (runtime lives here) ----------------
K_BETWEENNESS = 300          # sampled sources for approximate betweenness
BETWEENNESS_SEED = 42        # seed of the main sample
BETWEENNESS_SEED_B = 7       # seed of the second sample (noise check only)
RUN_STABILITY_CHECK = True   # False -> skip the second betweenness run
HARMONIC_SAMPLES = 300       # None -> exact harmonic centrality (very slow)
TOP_N = 15                   # rows per ranking table / bars per chart
CRITICAL_QUANTILE = 0.90     # the p90 rule we stress-test in section 11

# ---------------- stage folders ----------------
STAGE = OUT / "04_centrality_analysis"
TABLES = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
STAGE02 = OUT / "02_graph_construction"

print("stage folder :", STAGE)
print("reads stage  :", STAGE02)
print("networkx", nx.__version__, "| pandas", pd.__version__)

## 3. תוויות בעברית ב-matplotlib

שמות התחנות ב-GTFS feed הישראלי הם בעברית. ‏matplotlib שומר טקסט בסדר לוגי ואינו מיישם
את אלגוריתם ה-bidirectional של Unicode, ולכן תווית בעברית מצוירת משמאל לימין, כלומר
הפוכה ויזואלית. ה-patch שלהלן עוטף את `Text.set_text` וממיר מחרוזות עבריות לסדר תצוגה
פעם אחת, לפני שמצויר דבר. טקסט לטיני מוחזר ללא שינוי, וה-patch הוא אידמפוטנטי כך
שהרצה חוזרת של התא אינה מזיקה.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. טעינת הגרף שהופק במחברת 02

שלב זה אינו בונה מחדש את הגרף מ-GTFS גולמי - זה תפקידה של מחברת 02. כאן אנו קוראים את
טבלת הצמתים (מאפייני התחנות) ואת טבלת הקשתות המכוונות (שורה אחת לכל מקטע
`from_stop -> to_stop` יחד עם מספר הנסיעות העושות בו שימוש), ובונים מחדש את שני
אובייקטי `networkx` הנדרשים:

- `D`, גרף מכוון שמשקל הקשת בו הוא תדירות הנסיעות במקטע. זהו האובייקט שעליו רץ
  PageRank, משום שהכיוון מהותי לזרימה.
- `G`, ההיטל הלא-מכוון שמשקל הקשת בו הוא סכום שני הכיוונים. עליו רצים degree,
  betweenness ו-harmonic centrality, משום שנוסע העובר בין שתי תחנות סמוכות אינו מושפע
  מהכיוון שבו נרשמה הקשת.

טוען הנתונים סובלני במכוון לגבי שמות קבצים ושמות עמודות (`weight` מול `trip_frequency`,
`source` מול `from_stop` וכן הלאה), כך שימשיך לעבוד גם אם מחברת 02 תשנה שם כלשהו. אם
תיקיית שלב 02 חסרה לחלוטין, נזרקת שגיאה ברורה ופעילה במקום יצירה שקטה של גרף ריק.

In [ ]:
def _find_artifact(stage_dir, names, keyword):
    """Locate a stage artifact by exact name, then by keyword, under <stage>/tables or <stage>."""
    for name in names:
        for cand in (stage_dir / "tables" / name, stage_dir / name):
            if cand.exists():
                return cand
    hits = sorted(p for p in stage_dir.rglob("*.csv") if keyword in p.name.lower())
    return hits[0] if hits else None


def _pick_column(df, candidates):
    """Return the first column of df matching one of candidates (case-insensitive)."""
    lower = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    return None


if not STAGE02.exists():
    raise FileNotFoundError(
        f"{STAGE02} missing - run notebook 02_graph_construction first."
    )

edges_path = _find_artifact(
    STAGE02,
    ["edges.csv", "graph_edges.csv", "edges_directed.csv", "directed_edges.csv"],
    "edge",
)
nodes_path = _find_artifact(
    STAGE02, ["nodes.csv", "graph_nodes.csv", "stops_nodes.csv"], "node"
)
if edges_path is None:
    raise FileNotFoundError(
        f"No edge table found under {STAGE02} - run notebook 02_graph_construction first."
    )
print("edges from:", edges_path)
print("nodes from:", nodes_path)

edges_df = pd.read_csv(edges_path, dtype=str, encoding="utf-8-sig")
src_col = _pick_column(edges_df, ["from_stop", "from_stop_id", "source", "from", "u"])
dst_col = _pick_column(edges_df, ["to_stop", "to_stop_id", "target", "to", "v"])
wgt_col = _pick_column(
    edges_df, ["trip_frequency", "weight", "frequency", "trips", "n_trips", "count"]
)
if src_col is None or dst_col is None:
    raise ValueError(f"Cannot identify endpoint columns in {edges_path}: {list(edges_df.columns)}")
weights = (
    pd.to_numeric(edges_df[wgt_col], errors="coerce").fillna(1.0)
    if wgt_col is not None
    else pd.Series(1.0, index=edges_df.index)
)

# Directed trip graph: weight = number of trips using the segment.
D = nx.DiGraph()
for u, v, w in zip(edges_df[src_col], edges_df[dst_col], weights):
    if u == v or pd.isna(u) or pd.isna(v):
        continue
    if D.has_edge(u, v):
        D[u][v]["weight"] += float(w)
    else:
        D.add_edge(u, v, weight=float(w))

# Undirected projection: weight = sum of both directions.
G = nx.Graph()
for u, v, data in D.edges(data=True):
    if G.has_edge(u, v):
        G[u][v]["weight"] += data["weight"]
    else:
        G.add_edge(u, v, weight=data["weight"])

# Station attributes (name, coordinates, region, metro) if notebook 02 saved them.
ATTR = {}
VISIT_COL = None
if nodes_path is not None:
    nodes_df = pd.read_csv(nodes_path, dtype=str, encoding="utf-8-sig")
    id_col = _pick_column(nodes_df, ["stop_id", "node", "id"])
    name_col = _pick_column(nodes_df, ["stop_name", "name"])
    lat_col = _pick_column(nodes_df, ["lat", "stop_lat", "latitude"])
    lon_col = _pick_column(nodes_df, ["lon", "stop_lon", "longitude"])
    reg_col = _pick_column(nodes_df, ["region", "district"])
    met_col = _pick_column(nodes_df, ["metro", "metropolitan"])
    VISIT_COL = _pick_column(
        nodes_df,
        ["stop_use_count", "visits", "stop_visits", "visit_count", "n_visits"],
    )
    if id_col is not None:
        for row in nodes_df.itertuples(index=False):
            rec = dict(zip(nodes_df.columns, row))
            sid = rec[id_col]
            ATTR[sid] = {
                "stop_name": rec.get(name_col, "") if name_col else "",
                "lat": pd.to_numeric(rec.get(lat_col), errors="coerce") if lat_col else np.nan,
                "lon": pd.to_numeric(rec.get(lon_col), errors="coerce") if lon_col else np.nan,
                "region": rec.get(reg_col, "") if reg_col else "",
                "metro": rec.get(met_col, "") if met_col else "",
                "stop_use_count": pd.to_numeric(rec.get(VISIT_COL), errors="coerce")
                if VISIT_COL
                else np.nan,
            }
        # Keep stations that stage 02 listed even if they ended up with no segment.
        G.add_nodes_from(ATTR.keys())
        D.add_nodes_from(ATTR.keys())

print(f"directed graph  : {D.number_of_nodes():,} nodes, {D.number_of_edges():,} edges")
print(f"undirected graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"attributes for  : {len(ATTR):,} stations"
      + (f" (visit counts in column '{VISIT_COL}')" if VISIT_COL else " (no visit-count column)"))

## 5. בדיקת שפיות והרכיב הקשיר הגדול ביותר

שני מדדי מרכזיות גלובליים - betweenness ו-harmonic centrality - הם בעלי משמעות רק בין
צמתים שיכולים אכן להגיע זה לזה, ולכן שניהם מחושבים על הרכיב הקשיר הגדול ביותר (LCC)
של `G`. תחנות מחוץ ל-LCC מקבלות ציון 0, שהוא הערך הכן: הן אינן נגישות מגוף הרשת, ולכן
אינן נמצאות על אף מסלול קצר ביותר בתוכה. אנו מדפיסים את גודל ה-LCC כדי שהקורא יידע איזה
חלק מהרשת מכוסה בפועל על ידי המדדים הגלובליים.

In [ ]:
components = sorted(nx.connected_components(G), key=len, reverse=True)
Gc = G.subgraph(components[0]).copy()

n_nodes = G.number_of_nodes()
print(f"connected components : {len(components):,}")
print(f"largest component    : {Gc.number_of_nodes():,} nodes "
      f"({Gc.number_of_nodes() / n_nodes:.1%} of the network), "
      f"{Gc.number_of_edges():,} edges")
print(f"isolated / off-LCC   : {n_nodes - Gc.number_of_nodes():,} stations")
print(f"average degree       : {2 * G.number_of_edges() / n_nodes:.2f}")
print(f"betweenness sampling : k = {min(K_BETWEENNESS, Gc.number_of_nodes()):,} sources "
      f"= {min(K_BETWEENNESS, Gc.number_of_nodes()) / Gc.number_of_nodes():.2%} of the LCC")

## 6. Degree ו-degree משוקלל

המדדים הזולים ביותר, ובסיס ההשוואה שאליו מושווה כל השאר.

- `degree` - לכמה תחנות שכנות שונות מחוברת התחנה. זה סופר הסתעפות *טופולוגית*:
  תחנה המשרתת מסדרונות תנועה רבים ושונים מקבלת ציון גבוה.
- `degree_centrality` - אותו מספר מחולק ב-`n - 1`, הנרמול של networkx.
- `weighted_degree` - סכום תדירויות הנסיעות על כל המקטעים הסמוכים, כלומר נפח השירות
  היומי הכולל העובר דרך התחנה. זה סופר *תנועה*, לא הסתעפות: תחנה על מסדרון יחיד בקו
  אוטובוס תדיר במיוחד מקבלת ציון גבוה.
- `in_degree`, `out_degree`, `in_weight`, `out_weight` - הגרסאות המכוונות, הנלקחות
  מתוך `D`. עבור תחנה באמצע מסדרון דו-כיווני ערכים אלה קרובים לסימטריים; אי-סימטריה
  גדולה מסמנת בדרך כלל תחנה סופית או לולאה חד-כיוונית.

כל אלה הם שאילתות מילון על הגרף ועלותם זניחה.

In [ ]:
degree = dict(G.degree())
weighted_degree = dict(G.degree(weight="weight"))
in_degree = dict(D.in_degree())
out_degree = dict(D.out_degree())
in_weight = dict(D.in_degree(weight="weight"))
out_weight = dict(D.out_degree(weight="weight"))

print("degree      : max", max(degree.values()), "| mean", round(np.mean(list(degree.values())), 2))
print("weighted deg: max", int(max(weighted_degree.values())),
      "| mean", round(np.mean(list(weighted_degree.values())), 1))

## 7. PageRank משוקלל על הגרף המכוון

PageRank ממדל נוסע המבצע הילוך אקראי לאורך הרשת, ובהסתברות `1 - alpha = 0.15` מתנייד
(teleport) לתחנה אקראית. תחנה מקבלת ציון גבוה כאשר מקטעים רבים *בעלי שירות תדיר*
מובילים אליה מתחנות שהן עצמן חשובות, ולכן PageRank הוא תפיסת חשיבות משוקללת-זרימה ולא
ספירה מקומית גרידא.

אנו מריצים אותו על הגרף ה**מכוון** `D` עם `weight="weight"` (תדירות נסיעות), כך
שמקטעים חד-כיווניים ושירות א-סימטרי נלקחים בחשבון. צמתים תלויים (dangling nodes) -
תחנות סופיות ללא מקטע יוצא - מקבלים חלוקה מחדש אחידה של הדירוג שלהם, וזהו הטיפול
הסטנדרטי.

ברירת המחדל של `pagerank` ב-networkx משתמשת ב-SciPy. אם SciPy אינו מותקן, אנו נסוגים
לאיטרציית חזקות בפייתון טהור (אותו מימוש המשמש ב-pipeline הסקריפטים של הפרויקט), כך
שהמחברת לעולם אינה נכשלת בסביבה מינימלית. כל אחד משני המסלולים עולה מספר שניות.

In [ ]:
def weighted_pagerank(graph, weight="weight", alpha=0.85, max_iter=100, tol=1.0e-6):
    """Weighted PageRank via power iteration, without requiring SciPy."""
    nodes = list(graph.nodes)
    n = len(nodes)
    if n == 0:
        return {}
    rank = {node: 1.0 / n for node in nodes}
    out_w = {
        node: sum(data.get(weight, 1.0) for _, _, data in graph.out_edges(node, data=True))
        for node in nodes
    }
    for _ in range(max_iter):
        previous = rank
        dangling = sum(previous[node] for node in nodes if out_w[node] == 0)
        base = (1.0 - alpha) / n + alpha * dangling / n
        rank = {node: base for node in nodes}
        for source in nodes:
            total = out_w[source]
            if total == 0:
                continue
            src_rank = previous[source]
            for _, target, data in graph.out_edges(source, data=True):
                rank[target] += alpha * src_rank * data.get(weight, 1.0) / total
        if sum(abs(rank[node] - previous[node]) for node in nodes) < n * tol:
            return rank
    return rank


t0 = time.time()
try:
    pagerank = nx.pagerank(D, alpha=0.85, weight="weight")
    engine = "networkx (SciPy)"
except Exception as exc:  # SciPy missing or convergence failure
    print("networkx pagerank unavailable ->", exc, "| falling back to pure Python")
    pagerank = weighted_pagerank(D, weight="weight")
    engine = "pure Python power iteration"

print(f"PageRank via {engine} in {time.time() - t0:.1f}s; "
      f"sum = {sum(pagerank.values()):.4f} (should be ~1.0)")

## 8. Betweenness centrality מקורב

Betweenness סופר את שיעור המסלולים הקצרים ביותר העוברים דרך תחנה. זהו המדד שתופס בפועל
*גישור*: תחנה עם שני שכנים בלבד עדיין יכולה להיות בעלת betweenness עצום אם היא הדרך
היחידה מאזור אחד בארץ לאחר. זו בדיוק התכונה שמעניינת ניתוח של "תחנות קריטיות".

זהו גם המדד היקר ביותר בהפרש ניכר. חישוב Brandes מדויק הוא `O(n * m)`, ועבור כ-30,000
צמתים וכ-40,000 קשתות משמעותו BFS מכל צומת - שעות של חישוב. לכן אנו משתמשים באומדן
המדגמי של networkx: בחירת `k` צמתי מקור אקראיים, הרצת צבירת Brandes מהמקורות הללו בלבד,
ושינוי קנה מידה בהתאם.

**יש לקרוא זאת לפני שימוש במספרים.** עם `K_BETWEENNESS = 300` אנו דוגמים כ-1% מהמקורות
האפשריים. האומדן חסר הטיה בתוחלת, אך בעל שונות ממשית: ראש הדירוג (גשרים ארציים, הנמצאים
על מסלולים קצרים ביותר כמעט *מכל* מקור) יציב, בעוד שהאמצע והזנב זזים באופן ניכר בין
מדגמים. סעיף 11 מודד בכמה. ה-betweenness מחושב על ה-LCC וללא משקלים - "קצר ביותר"
פירושו מספר המקטעים הקטן ביותר, ולא המהיר או התדיר ביותר - וזוהי אותה מוסכמה הנהוגה בכל
מקום אחר בפרויקט זה.

In [ ]:
k_eff = min(K_BETWEENNESS, Gc.number_of_nodes())
t0 = time.time()
betweenness = nx.betweenness_centrality(
    Gc, k=k_eff, seed=BETWEENNESS_SEED, normalized=True, weight=None
)
print(f"approximate betweenness: k={k_eff}, seed={BETWEENNESS_SEED}, "
      f"{time.time() - t0:.1f}s")
nonzero = sum(1 for v in betweenness.values() if v > 0)
print(f"stations with a non-zero estimate: {nonzero:,} of {len(betweenness):,} "
      f"({nonzero / len(betweenness):.1%}) - the rest were simply never on a sampled path")

## 9. Harmonic centrality (מדגמי) על הרכיב הקשיר הגדול ביותר

Harmonic centrality מסכם `1 / d(u, v)` על פני כל התחנות האחרות `v`. בשונה מ-closeness
הוא מוגדר היטב גם כאשר חלק מהזוגות אינם נגישים, והוא עונה על שאלה שונה מזו של
betweenness: לא "כמה מסלולים עוברים דרכי" אלא "כמה קרוב אני לכל השאר", כלומר נגישות ולא
גישור.

החישוב המדויק הוא שוב BFS אחד לכל צומת. לכן אנו משתמשים באותו תכסיס דגימה כמו ב-pipeline
הסקריפטים של הפרויקט: הגרלת `HARMONIC_SAMPLES` מקורות אקראיים, הרצת BFS מכל אחד מהם,
צבירת `1 / d` לכל יעד שאליו הגענו, ושינוי קנה המידה בפקטור `n_nodes / n_samples`. מכיוון
שהגרף לא-מכוון, המרחק סימטרי, ולכן דגימת מקורות היא אומדן חסר הטיה תקף לסכום על פני
היעדים. יש להגדיר `HARMONIC_SAMPLES = None` כדי לחשב במדויק (איטי). כמו betweenness, גם
הגרסה המדגמית רועשת, אך harmonic centrality הוא גודל חלק וללא זנב כבד, ולכן הוא מתנהג
טוב בהרבה תחת דגימה מאשר betweenness.

In [ ]:
def approximate_harmonic_centrality(graph, samples, seed):
    """Estimate harmonic centrality from `samples` random BFS sources, rescaled to n."""
    if graph.number_of_nodes() == 0 or not samples:
        return {}
    rng = random.Random(seed)
    nodes = list(graph.nodes)
    sources = rng.sample(nodes, min(samples, len(nodes)))
    scores = defaultdict(float)
    for source in sources:
        lengths = nx.single_source_shortest_path_length(graph, source)
        for target, distance in lengths.items():
            if target == source or distance == 0:
                continue
            scores[target] += 1.0 / distance
    scale = graph.number_of_nodes() / len(sources)
    return {node: value * scale for node, value in scores.items()}


t0 = time.time()
if HARMONIC_SAMPLES is None:
    harmonic = nx.harmonic_centrality(Gc)
    harmonic_mode = "exact"
else:
    harmonic = approximate_harmonic_centrality(Gc, HARMONIC_SAMPLES, BETWEENNESS_SEED)
    harmonic_mode = f"sampled ({HARMONIC_SAMPLES} sources)"
print(f"harmonic centrality [{harmonic_mode}] in {time.time() - t0:.1f}s "
      f"for {len(harmonic):,} stations")

## 10. הרכבת `stop_metrics.csv` וטבלאות ה-top-N

כל מה שחושב עד כה ממופתח לפי `stop_id`; תא זה מאחד את הכול לטבלה מסודרת אחת יחד עם
מאפייני התחנות (שם, קואורדינטות, אזור, מטרופולין) וכותב אותה לדיסק. תחנות מחוץ ל-LCC
מקבלות `0.0` עבור betweenness ועבור harmonic centrality, כפי שהוסבר בסעיף 5.

בנוסף אנו כותבים טבלת top-`TOP_N` אחת לכל מדד. אלה הטבלאות שהדוח מצטט, ושמירתן כקבצים
נפרדים מבהירה מאיזה דירוג הגיעה כל טענה.

In [ ]:
default_attr = {"stop_name": "", "lat": np.nan, "lon": np.nan,
                "region": "", "metro": "", "stop_use_count": np.nan}

rows = []
for stop_id in G.nodes:
    a = ATTR.get(stop_id, default_attr)
    deg = degree.get(stop_id, 0)
    rows.append({
        "stop_id": stop_id,
        "stop_name": a.get("stop_name", ""),
        "region": a.get("region", ""),
        "metro": a.get("metro", ""),
        "lat": a.get("lat", np.nan),
        "lon": a.get("lon", np.nan),
        "stop_use_count": a.get("stop_use_count", np.nan),
        "degree": deg,
        "degree_centrality": deg / (n_nodes - 1) if n_nodes > 1 else 0.0,
        "weighted_degree": weighted_degree.get(stop_id, 0.0),
        "in_degree": in_degree.get(stop_id, 0),
        "out_degree": out_degree.get(stop_id, 0),
        "in_weight": in_weight.get(stop_id, 0.0),
        "out_weight": out_weight.get(stop_id, 0.0),
        "pagerank": pagerank.get(stop_id, 0.0),
        "approx_betweenness": betweenness.get(stop_id, 0.0),
        "approx_harmonic": harmonic.get(stop_id, 0.0),
        "in_largest_component": stop_id in Gc,
    })

metrics = pd.DataFrame(rows).sort_values("weighted_degree", ascending=False)
metrics["label"] = metrics["stop_name"].fillna("").astype(str).str.strip()
metrics["label"] = metrics["label"].where(metrics["label"] != "", metrics["stop_id"])

metrics.drop(columns=["label"]).to_csv(
    TABLES / "stop_metrics.csv", index=False, encoding="utf-8-sig"
)

RANKINGS = {
    "top_degree.csv": "degree",
    "top_weighted_degree.csv": "weighted_degree",
    "top_pagerank.csv": "pagerank",
    "top_approx_betweenness.csv": "approx_betweenness",
    "top_approx_harmonic.csv": "approx_harmonic",
}
for filename, column in RANKINGS.items():
    (metrics.drop(columns=["label"])
            .sort_values(column, ascending=False)
            .head(TOP_N)
            .to_csv(TABLES / filename, index=False, encoding="utf-8-sig"))

print(f"stop_metrics.csv written: {len(metrics):,} rows -> {TABLES}")
display(metrics.nlargest(10, "approx_betweenness")[
    ["stop_name", "stop_id", "region", "degree", "weighted_degree",
     "pagerank", "approx_betweenness"]
])

## 11. עד כמה רועש ה-betweenness המדגמי? (סעיף היושרה)

דירוג שימושי רק אם הוא בר-שחזור. כאן אנו מריצים מחדש את *אותו* אומדן עם seed אקראי שונה -
דבר ברשת אינו משתנה, רק אילו 300 מקורות הוגרלו - ומשווים את שתי התוצאות בשלוש דרכים:

1. **מתאם Spearman** בין שני וקטורי הציונים. מתאם כולל גבוה הוא צפוי ואינו מוכיח הרבה,
   משום שרוב התחנות מקבלות ציון של בערך 0 בשתי ההרצות.
2. **חפיפת top-50.** כמה מ-50 התחנות המובילות בהרצה A מופיעות גם ב-50 המובילות בהרצה B.
   זהו המספר המשמעותי עבור השאלה "אילו תחנות הן קריטיות".
3. **הסכמה על סף p90.** כלל נפוץ במורד הזרם הוא "תחנה היא קריטית אם ה-betweenness שלה
   מעל האחוזון ה-90". אנו בונים דגל זה מכל הרצה ומדווחים את חפיפת Jaccard בין שתי קבוצות
   הדגלים. כל אי-יציבות המתגלה כאן נורשת על ידי כל טענה הנבנית על סף כזה.

תא זה עולה הרצת betweenness נוספת אחת (כ-1-3 דקות). יש להגדיר `RUN_STABILITY_CHECK = False`
כדי לדלג עליו.

In [ ]:
stability = None
if RUN_STABILITY_CHECK:
    t0 = time.time()
    betweenness_b = nx.betweenness_centrality(
        Gc, k=k_eff, seed=BETWEENNESS_SEED_B, normalized=True, weight=None
    )
    print(f"second betweenness run (seed={BETWEENNESS_SEED_B}) in {time.time() - t0:.1f}s")

    joint = pd.DataFrame({
        "seed_a": pd.Series(betweenness),
        "seed_b": pd.Series(betweenness_b),
    }).fillna(0.0)

    rho_all = joint["seed_a"].corr(joint["seed_b"], method="spearman")
    both_positive = joint[(joint["seed_a"] > 0) & (joint["seed_b"] > 0)]
    rho_pos = both_positive["seed_a"].corr(both_positive["seed_b"], method="spearman")

    top_a = set(joint.nlargest(50, "seed_a").index)
    top_b = set(joint.nlargest(50, "seed_b").index)
    overlap50 = len(top_a & top_b) / 50
    top_a10 = set(joint.nlargest(10, "seed_a").index)
    top_b10 = set(joint.nlargest(10, "seed_b").index)
    overlap10 = len(top_a10 & top_b10) / 10

    thr_a = joint["seed_a"].quantile(CRITICAL_QUANTILE)
    thr_b = joint["seed_b"].quantile(CRITICAL_QUANTILE)
    flag_a = joint["seed_a"] > thr_a
    flag_b = joint["seed_b"] > thr_b
    union = int((flag_a | flag_b).sum())
    jaccard = int((flag_a & flag_b).sum()) / union if union else float("nan")

    stability = pd.DataFrame([{
        "k_samples": k_eff,
        "lcc_nodes": Gc.number_of_nodes(),
        "sampling_fraction": k_eff / Gc.number_of_nodes(),
        "spearman_all_nodes": round(float(rho_all), 4),
        "spearman_both_positive": round(float(rho_pos), 4),
        "top10_overlap": overlap10,
        "top50_overlap": overlap50,
        "p90_flag_jaccard": round(float(jaccard), 4),
        "p90_flagged_seed_a": int(flag_a.sum()),
        "p90_flagged_seed_b": int(flag_b.sum()),
    }])
    stability.to_csv(TABLES / "betweenness_stability.csv", index=False, encoding="utf-8-sig")
    display(stability.T.rename(columns={0: "value"}))

    fig, ax = plt.subplots(figsize=(6.5, 6))
    ax.scatter(joint["seed_a"], joint["seed_b"], s=4, alpha=0.25, color="#dc2626")
    lim = max(joint["seed_a"].max(), joint["seed_b"].max()) * 1.05
    ax.plot([0, lim], [0, lim], color="#334155", lw=1, ls="--", label="perfect agreement")
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_xlabel(f"Approx. betweenness (seed {BETWEENNESS_SEED})")
    ax.set_ylabel(f"Approx. betweenness (seed {BETWEENNESS_SEED_B})")
    ax.set_title(f"Same estimator, different sample (k={k_eff})")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES / "betweenness_seed_stability.png", dpi=150)
    plt.show()
else:
    print("RUN_STABILITY_CHECK is False - skipping the betweenness noise measurement.")

## 12. האם המדדים מסכימים? מטריצת מתאם Spearman

מתאם Spearman (מתאם דירוגים) הוא הכלי הנכון כאן משום שכל ההתפלגויות הללו בעלות זנב כבד:
Pearson היה נשלט על ידי קומץ התחנות העצומות. מה שאנו רוצים לדעת הוא האם המדדים מייצרים
את אותו *סדר*.

פרשנות שכדאי לזכור בעת קריאת המטריצה:
- `degree` ו-`weighted_degree` מתואמים חזק, אך אינם אותו דבר - האחד הוא הסתעפות והשני
  הוא תנועה.
- `pagerank` הוא במידה רבה גרסה מוחלקת של in-degree משוקלל, ולכן מתאם גבוה עם
  `weighted_degree` הוא צפוי ו*אינו* תגלית עצמאית.
- התא המעניין הוא `approx_betweenness` מול המדדים המקומיים. מתאם בינוני ולא גבוה הוא
  התוצאה המהותית: משמעותו שתחנות מגשרות אינן פשוט התחנות העמוסות ביותר. יש לזכור
  שעמודת ה-betweenness היא האומדן המדגמי הרועש, ולכן המתאמים שלה מוחלשים (attenuated) -
  המתאם האמיתי עם betweenness מדויק היה גבוה במקצת.

In [ ]:
CORR_COLS = ["degree", "weighted_degree", "pagerank", "approx_betweenness", "approx_harmonic"]
corr = metrics[CORR_COLS].astype(float).corr(method="spearman").round(4)
corr.to_csv(TABLES / "centrality_correlation_spearman.csv", encoding="utf-8-sig")
display(corr)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(CORR_COLS)))
ax.set_xticklabels([c.replace("_", " ") for c in CORR_COLS], rotation=35, ha="right")
ax.set_yticks(range(len(CORR_COLS)))
ax.set_yticklabels([c.replace("_", " ") for c in CORR_COLS])
for i in range(len(CORR_COLS)):
    for j in range(len(CORR_COLS)):
        value = corr.values[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center",
                color="white" if abs(value) > 0.6 else "black", fontsize=10)
ax.set_title("Spearman correlation between centrality measures")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(FIGURES / "centrality_correlation_heatmap.png", dpi=150)
plt.show()

## 13. התחנות המובילות לכל מדד

תרשים עמודות אופקי אחד לכל מדד, המציג את `TOP_N` התחנות בעלות הציון הגבוה ביותר עם שמותיהן
העבריים (מוצגים דרך ה-bidi patch מסעיף 3; היכן ששם חסר אנו נסוגים ל-`stop_id` הגולמי).
העמודות מצוירות במיקומי y מפורשים ולא לפי תווית קטגוריה, כך ששתי תחנות שונות בעלות אותו
שם אינן מתמזגות לעמודה אחת - סכנה ממשית כאן, שכן שמות כמו אותו רחוב בערים שונות חוזרים על
עצמם לאורך ה-feed.

ההשוואה בין ארבעת התרשימים היא העיקר: אם אותן תחנות שולטות בכל תרשים, המדדים יתירים; ואם
תרשים ה-betweenness מונה תחנות שאינן מופיעות בשום מקום אחר, הרי שברשת קיימים צווארי בקבוק
מבניים שנפח התנועה לבדו היה מפספס.

In [ ]:
def plot_top(df, column, title, color, filename, n=TOP_N):
    top = df.nlargest(n, column)[["label", column]].copy().sort_values(column)
    y = np.arange(len(top))
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(y, top[column].astype(float).values, color=color)
    ax.set_yticks(y)
    ax.set_yticklabels(top["label"].astype(str).values)
    ax.set_xlabel(column.replace("_", " ").title())
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(FIGURES / filename, dpi=150)
    plt.show()


plot_top(metrics, "degree",
         f"Top {TOP_N} stations by degree (number of neighbouring stops)",
         "#2563eb", "top_degree.png")
plot_top(metrics, "weighted_degree",
         f"Top {TOP_N} stations by weighted degree (daily service volume)",
         "#0891b2", "top_weighted_degree.png")
plot_top(metrics, "pagerank",
         f"Top {TOP_N} stations by weighted PageRank",
         "#16a34a", "top_pagerank.png")
plot_top(metrics, "approx_betweenness",
         f"Top {TOP_N} stations by approximate betweenness (k={k_eff}, noisy)",
         "#dc2626", "top_approx_betweenness.png")
plot_top(metrics, "approx_harmonic",
         f"Top {TOP_N} stations by harmonic centrality ({harmonic_mode})",
         "#d97706", "top_approx_harmonic.png")

## 14. היכן המדדים חלוקים: תרשימי פיזור ומפה

מטריצת המתאמים נותנת מספר אחד לכל זוג; תרשימים אלה מראים את הצורה שמאחוריו.

- *Degree מול betweenness*: יש לחפש נקודות נמוכות על ציר ה-x וגבוהות על ציר ה-y. אלה
  התחנות בעלות הקישוריות הנמוכה והגישור הגבוה - אלה שניתוח חוסן (resilience) צריך לדאוג
  להן, ואלה שדירוג מבוסס degree היה מפספס לחלוטין.
- *Degree מול PageRank*: צפוי להיות הדוק בהרבה, שכן PageRank הוא בעצם degree משוקלל-זרימה.
- המפה מציירת את כל התחנות בגוון חיוור ומדגישה את 50 התחנות בעלות ה-betweenness הגבוה
  ביותר, צבועות לפי ציון. אם נקודות אלה מסתדרות לאורך מסדרונות תל אביב - ירושלים - חיפה
  ולא מתפזרות, האומדן קולט מבנה ארצי אמיתי ולא רק רעש דגימה - בדיקה איכותנית מועילה
  לסעיף 11.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(metrics["degree"], metrics["approx_betweenness"],
                s=4, alpha=0.3, color="#2563eb")
axes[0].set_xlabel("Degree")
axes[0].set_ylabel("Approx. betweenness")
axes[0].set_title("Degree vs betweenness (disagreement is the interesting part)")
axes[1].scatter(metrics["degree"], metrics["pagerank"],
                s=4, alpha=0.3, color="#16a34a")
axes[1].set_xlabel("Degree")
axes[1].set_ylabel("PageRank")
axes[1].set_yscale("log")
axes[1].set_title("Degree vs PageRank (log scale)")
fig.tight_layout()
fig.savefig(FIGURES / "centrality_scatter.png", dpi=150)
plt.show()

geo = metrics.dropna(subset=["lat", "lon"]).copy()
if len(geo):
    top_geo = geo.nlargest(50, "approx_betweenness")
    fig, ax = plt.subplots(figsize=(7.5, 10))
    ax.scatter(geo["lon"], geo["lat"], s=2, alpha=0.15, color="#94a3b8", label="all stations")
    sc = ax.scatter(top_geo["lon"], top_geo["lat"], s=45,
                    c=top_geo["approx_betweenness"], cmap="Reds",
                    edgecolors="black", linewidths=0.3, zorder=5)
    fig.colorbar(sc, ax=ax, label="Approx. betweenness", shrink=0.7)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Top 50 stations by approximate betweenness")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(FIGURES / "betweenness_map.png", dpi=150)
    plt.show()
else:
    print("No coordinates available in the stage-02 node table - skipping the map.")

## 15. Degree משוקלל מול מספר הביקורים בתחנה - טאוטולוגיה, לא ממצא

מפתה לדווח "weighted degree מתואם בכ-0.99 עם מספר הפעמים שנסיעה עוצרת בתחנה, ולכן תחנות
עמוסות הן מרכזיות". המתאם הזה אמיתי אך הוא **נכון מעצם הבנייה ואינו נושא מידע כלשהו**.

שני הגדלים נבנים מאותן שורות של `stop_times.txt`. בכל פעם שנסיעה עוצרת בתחנה `s` באמצע
מסלולה היא תורמת בדיוק מקטע נכנס אחד ומקטע יוצא אחד, ולכן

```
weighted_degree(s) = in_weight(s) + out_weight(s) ~= 2 * visits(s)
```

עם סטיות רק בקצות הנסיעה (תחנה סופית תורמת צד אחד ולא שניים) ובכניסות תחנה כפולות
עוקבות, שבונה הגרף משמיט. כך שהמתאם הקרוב ל-1 הוא ניסוח מחדש של ההגדרה, בקירוב
`y ~= 2x`, ואינו יכול לשמש ראיה לדבר בנוגע לרשת.

התא שלהלן מאמת במדויק את הזהות `weighted_degree == in_weight + out_weight`, ומדווח את
המתאם עם מספר הביקורים *רק* אם שלב 02 שמר במקרה עמודת ספירת ביקורים (או אם הקובץ
`stop_times.txt` בגודל 816 MB כבר נמצא על הדיסק - אנו לעולם לא מורידים אותו רק לצורך
בדיקה זו). כך או כך, המספר מודפס כבדיקת שפיות ולא כתוצאה.

In [ ]:
# 1) Exact identity: undirected weighted degree = in_weight + out_weight.
identity_gap = (metrics["weighted_degree"]
                - (metrics["in_weight"] + metrics["out_weight"])).abs().max()
print(f"max |weighted_degree - (in_weight + out_weight)| = {identity_gap:.6f} "
      "(0 means the two are the same quantity)")

# 2) Visit counts, only if they are already available - no 816 MB download here.
visits = metrics["stop_use_count"]
if visits.notna().sum() == 0:
    stop_times_path = DATA / "stop_times.txt"
    if stop_times_path.exists():
        print("Counting stop visits from the local stop_times.txt (one streaming pass) ...")
        csv.field_size_limit(10_000_000)
        counts = Counter()
        with open(stop_times_path, encoding="utf-8-sig") as handle:
            reader = csv.reader(handle)
            header = next(reader)
            si = header.index("stop_id")
            for row in reader:
                counts[row[si]] += 1
        visits = metrics["stop_id"].map(counts)
    else:
        visits = pd.Series(np.nan, index=metrics.index)

if visits.notna().sum() > 0:
    paired = pd.DataFrame({"weighted_degree": metrics["weighted_degree"].astype(float),
                           "visits": visits.astype(float)}).dropna()
    rho = paired["weighted_degree"].corr(paired["visits"], method="spearman")
    ratio = (paired["weighted_degree"] / paired["visits"].replace(0, np.nan)).median()
    print(f"Spearman(weighted_degree, visits) = {rho:.4f} over {len(paired):,} stations")
    print(f"median weighted_degree / visits   = {ratio:.3f}  (~2 confirms the identity above)")
    print("Reminder: this is a definitional relationship, not an empirical finding.")
else:
    print("No visit-count column from stage 02 and no local stop_times.txt - "
          "skipping the numeric check. The identity in the markdown above still holds.")

## 16. מסקנות

יש לקרוא מסקנות אלה יחד עם הטבלאות שב-`outputs/nb/04_centrality_analysis/tables/`;
המספרים המדויקים תלויים בתמונת המצב של ה-feed ובזרעי הדגימה (seeds).

1. **המדדים אינם ברי-החלפה.** ‏Degree, degree משוקלל ו-PageRank מדרגים תחנות לפי כמות
   השירות העוברת דרכן, ומסכימים במידה רבה זה עם זה. ‏Betweenness מדרג אותן לפי כמה
   מקישוריות הרשת תלויה בהן, ומסכים פחות באופן ניכר. התחנות הממוקמות בתרשים הפיזור באזור
   degree נמוך / betweenness גבוה הן אלה שניתוח חוסן צריך להתעניין בהן, ודירוג מבוסס
   תנועה לעולם לא היה חושף אותן.

2. **עמודת ה-betweenness היא מדגם של 1% ויש להתייחס אליה ככזו.** עם `K_BETWEENNESS = 300`
   מקורות שנדגמו מתוך רכיב בן כ-30,000 צמתים, האומדן חסר הטיה בתוחלת אך רועש ברמת
   התצפית הבודדת. סעיף 11 מכמת זאת באמצעות הרצה חוזרת עם seed שני: ראש הדירוג יציב באופן
   סביר (תחנות המסדרונות הארציים נמצאות על מסלולים קצרים ביותר כמעט מכל מקור), אך
   ההשתייכות ל"top 50" הרחב יותר, ובמיוחד לקבוצת ה"קריטיות" לפי סף p90, זזה בין מדגמים.
   **כל כלל במורד הזרם מהצורה "betweenness מעל האחוזון ה-90 פירושו קריטי" יורש את
   אי-היציבות הזו**, ויש להתייחס לסף כאל רצועה מטושטשת ולא כאל קו חד. העלאת
   `K_BETWEENNESS` מקטינה את השונות בקירוב כ-`1/sqrt(k)` בעלות לינארית; החישוב המדויק היה
   מבטל אותה לחלוטין במחיר של שעות.

3. **Degree משוקלל מול מספר הביקורים בתחנה אינו ממצא.** כפי שהוצג בסעיף 15,
   `weighted_degree = in_weight + out_weight ~= 2 * visits` מעצם הבנייה, כששניהם נגזרים
   מאותן שורות GTFS. המתאם של כ-0.99 מנסח מחדש הגדרה. ראוי לדווח עליו רק כבדיקת תקינות
   נתונים לכך שהגרף נבנה נכון.

4. **Harmonic centrality מוסיף נקודת מבט שלישית ונבדלת** - נגישות ולא נפח או גישור - אך
   הוא חלק וצפוי מבחינה גיאוגרפית: הוא מתגמל בעיקר קרבה למרכז אשכול גוש דן הצפוף. גם הוא
   נאמד כאן ממקורות שנדגמו, אם כי הוא רגיש הרבה פחות לדגימה מאשר betweenness, משום שהוא
   ממצע על פני מרחקים רבים במקום להתרכז במסלולים קצרים ביותר נדירים.

5. **הסתייגות לגבי כיסוי.** ‏Betweenness ו-harmonic centrality מוגדרים רק על הרכיב הקשיר
   הגדול ביותר; תחנות מחוצה לו נרשמות כ-0.0. זהו הערך הנכון עבור מדדים אלה, אך משמעות
   הדבר היא שעמודת האפסים מערבבת שני מצבים שונים מאוד - "מעולם לא נדגמה" ו"לא נמצאת ברשת
   הראשית" - והדגל `in_largest_component` שב-`stop_metrics.csv` הוא זה שמבחין ביניהם.